In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp feature_engineering

# Feature engineering
> Create exogenous regressors for your models

In [ ]:
#| export
from typing import Tuple

import numpy as np

import utilsforecast.processing as ufp
from utilsforecast.compat import DataFrame
from utilsforecast.validation import validate_format, validate_freq

In [ ]:
#| export
def fourier(
    df: DataFrame,
    freq: str,
    season_length: int,
    k: int,
    h: int = 0,
    id_col: str = 'unique_id',
    time_col: str = 'ds',
) -> Tuple[DataFrame, DataFrame]:
    """Compute fourier seasonal terms for training and forecasting

    Parameters
    ----------
    df : pandas or polars DataFrame
        Dataframe with ids, times and values for the exogenous regressors.
    freq : str or int
        Frequency of the data. Must be a valid pandas or polars offset alias, or an integer.
    season_length : int
        Number of observations per unit of time. Ex: 24 Hourly data.
    k : int
        Maximum order of the fourier terms
    h : int (default=0)
        Forecast horizon.        
    id_col : str (default='unique_id')
        Column that identifies each serie.
    time_col : str (default='ds')
        Column that identifies each timestep, its values can be timestamps or integers.

    Returns
    -------
    transformed_df : pandas or polars DataFrame
        Original DataFrame with the computed features
    future_df : pandas or polars DataFrame
        DataFrame with future values
    """
    if not isinstance(h, int) or h < 0:
        raise ValueError('`h` must be a non-negative integer')
    validate_format(df, id_col, time_col, None)
    validate_freq(df[time_col], freq)

    # compute terms based on the max size
    id_counts = ufp.counts_by_id(df, id_col)
    uids = id_counts[id_col]
    sizes = id_counts['counts'].to_numpy()
    max_size = sizes.max()
    # taken from: https://github.com/tblume1992/TSUtilities/blob/main/TSUtilities/TSFeatures/fourier_seasonality.py
    x = 2 * np.pi * np.arange(1, k + 1) / season_length
    x = x.astype(np.float32)
    t = np.arange(1, max_size + 1 + h, dtype=np.float32)
    x = x * t[:, None]
    terms = np.hstack([np.sin(x), np.cos(x)])
    cols = [f'{op}{i+1}_{season_length}' for op in ('sin', 'cos') for i in range(k)]

    # slice for each serie
    vals = np.empty((sizes.sum(), 2 * k), dtype=np.float32)
    future_vals = np.empty((h * sizes.size, 2 * k), dtype=np.float32)
    start = 0
    for i, size in enumerate(sizes):
        vals[start : start + size, :] = terms[:size]
        future_vals[i * h: (i + 1) * h] = terms[size : size + h]
        start += size

    # assign back to df
    sort_idxs = ufp.maybe_compute_sort_indices(df, id_col, time_col)
    times = df[time_col]
    if sort_idxs is not None:
        restore_idxs = np.empty_like(sort_idxs)
        restore_idxs[sort_idxs] = np.arange(sort_idxs.size)
        vals = vals[restore_idxs]
        times = ufp.take_rows(times, sort_idxs)
    last_times = ufp.take_rows(times, sizes.cumsum() - 1)
    df = ufp.copy_if_pandas(df, deep=False)
    transformed = ufp.assign_columns(df, cols, vals)

    if h == 0:
        return transformed, type(df)({})

    # future vals
    future_df = ufp.make_future_dataframe(
        uids=uids,
        last_times=last_times,
        freq=freq,
        h=h,
        id_col=id_col,
        time_col=time_col,
    )
    future_df = ufp.assign_columns(future_df, cols, future_vals)
    return transformed, future_df

In [ ]:
import pandas as pd

from utilsforecast.data import generate_series

In [ ]:
series = generate_series(5)
transformed_df, future_df = fourier(series, freq='D', season_length=7, k=2, h=1)
transformed_df

In [ ]:
future_df

In [ ]:
#| hide
transformed_df2, future_df2 = fourier(series.sample(frac=1.0), freq='D', season_length=7, k=2, h=1)
pd.testing.assert_frame_equal(
    transformed_df,
    transformed_df2.sort_values(['unique_id', 'ds']).reset_index(drop=True)
)
pd.testing.assert_frame_equal(future_df, future_df2)

In [ ]:
#| hide
#| polars
import polars as pl
import polars.testing

In [ ]:
#| hide
#| polars
series_pl = generate_series(5, engine='polars')
transformed_pl, future_pl = fourier(series_pl, freq='1d', season_length=7, k=2, h=1)
transformed_pl2, future_pl2 = fourier(series_pl.sample(fraction=1.0), freq='1d', season_length=7, k=2, h=1)
pl.testing.assert_frame_equal(transformed_pl, transformed_pl2)
pl.testing.assert_frame_equal(future_pl, future_pl2)
pd.testing.assert_frame_equal(
    transformed_df.drop(columns=['unique_id']),
    transformed_pl.drop(columns=['unique_id']).to_pandas()
)
pd.testing.assert_frame_equal(
    future_df.drop(columns=['unique_id']),
    future_pl.drop(columns=['unique_id']).to_pandas()
)